# Question-to-Cypher (Q2C) — E2E Evaluation

Notebook untuk mengevaluasi pipeline E2E: **Question → Cypher → Neo4j → Answer**.

**Alur:**
1. Load test data dari Google Sheets / CSV (berisi `TEST_ID`, `QUESTION`)
2. Kirim setiap pertanyaan ke backend `/api/qa`
3. Capture: `GENERATED_CYPHER`, `CYPHER_QUERY_RESULT`, `ANSWER`, `STATUS`
4. Hitung pass rate per kategori
5. Tulis hasil ke Google Sheets

**Perbedaan dari `04_evaluation.ipynb` (KG Extraction):**
- KG Extraction: query Cypher sudah ada di test data, hanya execute + compare
- Q2C E2E: **question saja** → backend generate Cypher → execute → jawab

In [53]:
# === Setup ===
import sys
import json
import os
import time
import requests
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
from neo4j import GraphDatabase

PROJECT_ROOT = Path(os.getcwd()).parent.parent if 'notebooks' in str(Path(os.getcwd())) else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
load_dotenv()

print(f'Project root: {PROJECT_ROOT}')

Project root: d:\TA\llm-driven-legal-kg-visualization


## Step 0: Configuration

Set `EXPERIMENT_ID`, backend URL, dan sumber test data.

**Experiment ID format:** `Q2C_{NUMBER}`

In [54]:
# === Configuration ===
EXPERIMENT_ID = "Q2C_005"              # Unique ID per eksperimen
DOCUMENT_IDS = ["POJK_11_2022", "UU_11_2008", "UU_19_2016"]          # doc_ids untuk query (list)

# Backend API
BACKEND_URL = "http://localhost:8000"   # Backend FastAPI base URL
API_TIMEOUT = 120                       # Timeout per request (seconds)
DELAY_BETWEEN_REQUESTS = 1              # Delay antar request (seconds)

# Sumber test data: 'csv' atau 'gsheets'
TEST_DATA_SOURCE = "gsheets"
CSV_PATH = "HOTS_Q2C_TEST_DATA.csv"              # Jika source = csv
GSHEETS_SHEET_NAME = "UU_11_2008_E2E_DATATEST_V3"   # Jika source = gsheets

# Tulis hasil ke Google Sheets?
WRITE_TO_GSHEETS = False
EXPERIMENT_SHEET_NAME = f"EXP_E2E_{GSHEETS_SHEET_NAME}_{'_'.join(DOCUMENT_IDS)}"    # 1 sheet per eksperimen

print(f'Experiment: {EXPERIMENT_ID}')
print(f'Document IDs: {DOCUMENT_IDS}')
print(f'Backend: {BACKEND_URL}')
print(f'Test data source: {TEST_DATA_SOURCE}')
print(f'Output sheet: {EXPERIMENT_SHEET_NAME}')

Experiment: Q2C_005
Document IDs: ['POJK_11_2022', 'UU_11_2008', 'UU_19_2016']
Backend: http://localhost:8000
Test data source: gsheets
Output sheet: EXP_E2E_UU_11_2008_E2E_DATATEST_V3_POJK_11_2022_UU_11_2008_UU_19_2016


## Step 1: Load Test Data

In [55]:
# === Load Test Data ===
if TEST_DATA_SOURCE == "gsheets":
    from modules.google_sheets_utils import GoogleUtil
    gu = GoogleUtil(
        private_key=os.getenv('GOOGLE_SHEETS_PRIVATE_KEY', ''),
        client_email=os.getenv('GOOGLE_SHEETS_CLIENT_EMAIL', '')
    )
    spreadsheet_id = os.getenv('GOOGLE_SPREADSHEET_ID', '')
    test_df = gu.load_dataframe_from_sheet(spreadsheet_id, GSHEETS_SHEET_NAME)
else:
    # Look in project root, parent directory, or data directory
    if os.path.exists(CSV_PATH):
        test_df = pd.read_csv(CSV_PATH)
    elif os.path.exists(PROJECT_ROOT / CSV_PATH):
        test_df = pd.read_csv(PROJECT_ROOT / CSV_PATH)
    else:
        test_df = pd.read_csv(PROJECT_ROOT.parent / CSV_PATH)

print(f'Loaded {len(test_df)} test cases')
print(f'Columns: {list(test_df.columns)}')
test_df[['TEST_ID', 'QUESTION', 'CATEGORY', 'EXPECTED_CYPHER_QUERY']].head(5)

2026-06-01 11:16:35,668 - INFO - Retrieving worksheet 'UU_11_2008_E2E_DATATEST_V3' from spreadsheet ID '1oN5kMN_OI8WyITAQgJ3-S_0GlzraXug8p2tMKSmq7u0'...


2026-06-01 11:16:37,586 - INFO - Successfully loaded 178 rows from worksheet 'UU_11_2008_E2E_DATATEST_V3'.


Loaded 178 test cases
Columns: ['TEST_ID', 'CATEGORY', 'QUESTION']


KeyError: "['EXPECTED_CYPHER_QUERY'] not in index"

## Step 2: Verify Backend is Running

In [ ]:
# === Check Backend Health ===
try:
    resp = requests.get(f"{BACKEND_URL}/docs", timeout=5)
    print(f'✅ Backend is running at {BACKEND_URL} (status: {resp.status_code})')
except Exception as e:
    print(f'❌ Backend not reachable: {e}')
    print('Make sure to run: uvicorn app.main:app --reload --port 8000')

✅ Backend is running at http://localhost:8000 (status: 200)


In [ ]:
# === Connect to Neo4j (for re-executing generated Cypher) ===
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Test connection
with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run('RETURN 1 AS test')
    print(f'✅ Connected to Neo4j database: "{NEO4J_DATABASE}"')


✅ Connected to Neo4j database: "experiment-2"


## Step 3: Define E2E Evaluation Logic

In [ ]:
def call_qa_api(question: str, doc_ids: list, backend_url: str, timeout: int = 120) -> dict:
    """Call the /api/qa endpoint and extract all response fields.
    
    Returns:
        dict with keys: status, cypher, cypher_result, answer, error, time_s,
                        references, process_steps
    """
    start = time.time()
    try:
        resp = requests.post(
            f"{backend_url}/api/qa",
            json={"question": question, "doc_ids": doc_ids},
            timeout=timeout,
        )
        elapsed = time.time() - start
        
        if resp.status_code != 200:
            return {
                'status': 'HTTP_ERROR',
                'cypher': '',
                'cypher_result': '',
                'answer': '',
                'error': f'HTTP {resp.status_code}: {resp.text[:200]}',
                'time_s': round(elapsed, 2),
                'references': '',
                'process_steps': [],
            }
        
        data = resp.json()
        steps = data.get('process_steps', [])
        answer = data.get('answer', '')
        cypher = data.get('cypher_query', '')
        
        # Extract Cypher Query Result from process_steps
        # Step 3 = Neo4j execution result (contains 'detail' and 'data')
        cypher_result_detail = ''
        cypher_result_data = []
        for s in steps:
            if s.get('step') == 3:
                cypher_result_detail = s.get('detail', '')
                cypher_result_data = s.get('data', [])
                break
        
        # Format cypher result: combine detail + raw data
        if cypher_result_data:
            cypher_result = json.dumps(cypher_result_data, ensure_ascii=False)
        else:
            cypher_result = cypher_result_detail
        
        # Determine status
        cypher_ok = any(s.get('step') == 2 and s.get('status') == 'done' for s in steps)
        has_results = 'hasil ditemukan' in cypher_result_detail and not cypher_result_detail.startswith('0')
        has_answer = bool(answer) and len(answer) > 20
        
        if cypher_ok and has_results and has_answer:
            status = 'PASS'
        elif cypher_ok and not has_results:
            status = 'FAIL'
        elif not cypher_ok:
            status = 'FAIL'
        else:
            status = 'PARTIAL'
        
        # Extract metadata
        references = '; '.join(data.get('references', []))
        
        return {
            'status': status,
            'cypher': cypher,
            'cypher_result': cypher_result,
            'cypher_result_detail': cypher_result_detail,
            'answer': answer,
            'time_s': round(elapsed, 2),
            'references': references,
            'process_steps': steps,
        }
        
    except requests.exceptions.Timeout:
        return {
            'status': 'ERROR',
            'cypher': '',
            'cypher_result': '',
            'cypher_result_detail': '',
            'answer': '',
            'error': f'Timeout ({timeout}s)',
            'time_s': timeout,
            'references': '',
            'process_steps': [],
        }
    except Exception as e:
        elapsed = time.time() - start
        return {
            'status': 'ERROR',
            'cypher': '',
            'cypher_result': '',
            'cypher_result_detail': '',
            'answer': '',
            'error': str(e),
            'time_s': round(elapsed, 2),
            'references': '',
            'process_steps': [],
        }


print('✅ E2E evaluation functions defined')


def execute_cypher_on_neo4j(cypher_query: str) -> list:
    """Re-execute a Cypher query directly on Neo4j and return JSON results."""
    if not cypher_query or not cypher_query.strip():
        return []
    try:
        with driver.session(database=NEO4J_DATABASE) as session:
            result = session.run(cypher_query)
            records = [dict(record) for record in result]
            return records
    except Exception as e:
        return [{'error': str(e)}]


print('✅ Neo4j re-execution helper defined')


✅ E2E evaluation functions defined
✅ Neo4j re-execution helper defined


## Step 4: Run E2E Evaluation

In [ ]:
# === Run All Test Cases ===
import os
results = []
total = len(test_df)

# Create real-time log file path
os.makedirs('data/evaluation', exist_ok=True)
log_file_path = f'data/evaluation/e2e_eval_log_{EXPERIMENT_ID}.txt'

# Initialize log file
with open(log_file_path, 'w', encoding='utf-8') as log_f:
    log_f.write(f"=== Q2C Evaluation Log: {EXPERIMENT_ID} ===\n")
    log_f.write(f"Documents: {', '.join(DOCUMENT_IDS)}\n")
    log_f.write(f"Backend: {BACKEND_URL}\n")
    log_f.write("="*80 + "\n\n")

print(f"Real-time evaluation logs will be written to: {log_file_path}\n")

for idx, row in test_df.iterrows():
    test_id = row['TEST_ID']
    question = row['QUESTION']
    
    print(f"[{idx+1}/{total}] {test_id}: {question}")
    
    # Call backend API
    result = call_qa_api(
        question=question,
        doc_ids=DOCUMENT_IDS,
        backend_url=BACKEND_URL,
        timeout=API_TIMEOUT,
    )
    
    # Print status
    status_icon = {'PASS': '✅', 'FAIL': '❌', 'ERROR': '⚠️', 'PARTIAL': '🟡'}.get(result['status'], '❓')
    print(f"    Status: {status_icon} {result['status']} ({result['time_s']:.1f}s)")
    
    # Print FULL Cypher Query
    if result['cypher']:
        print("    Generated Cypher:")
        print("    " + "-"*80)
        for line in result['cypher'].splitlines():
            print(f"    {line}")
        print("    " + "-"*80)
    
    # === RE-EXECUTE generated Cypher directly on Neo4j ===
    cypher_json_result = []
    formatted_cypher_result = ''
    if result['cypher']:
        cypher_json_result = execute_cypher_on_neo4j(result['cypher'])
        if cypher_json_result:
            formatted_cypher_result = json.dumps(cypher_json_result, indent=2, ensure_ascii=False)
            print(f"    Neo4j Query Result: {len(cypher_json_result)} records returned")
            print("    " + "-"*80)
            # Show first 50 lines of formatted JSON
            lines = formatted_cypher_result.splitlines()
            for line in lines[:50]:
                print(f"    {line}")
            if len(lines) > 50:
                print(f"    ... ({len(lines) - 50} more lines)")
            print("    " + "-"*80)
        else:
            print("    Neo4j Query Result: 0 records (empty)")
    
    # Print original backend result detail
    if result.get('cypher_result_detail', ''):
        print(f"    Backend Result Detail: {result['cypher_result_detail']}")
        
    # Print FULL LLM Answer
    if result['answer']:
        print("    Final LLM Answer:")
        print("    " + "-"*80)
        for line in result['answer'].splitlines():
            print(f"    {line}")
        print("    " + "-"*80)
        
    print("\n" + "="*100 + "\n")
    
    # Write to log file in real time
    with open(log_file_path, 'a', encoding='utf-8') as log_f:
        log_f.write(f"[{idx+1}/{total}] {test_id}: {question}\n")
        log_f.write(f"Status: {result['status']} ({result['time_s']:.1f}s)\n")
        if result['cypher']:
            log_f.write(f"Generated Cypher:\n{result['cypher']}\n")
        if formatted_cypher_result:
            log_f.write(f"Cypher Query Result ({len(cypher_json_result)} records):\n{formatted_cypher_result}\n")
        if result['answer']:
            log_f.write(f"LLM Answer:\n{result['answer']}\n")
        if result.get('error', ''):
            log_f.write(f"Error:\n{result['error']}\n")
        log_f.write("-" * 80 + "\n\n")
    
    # Build result row
    results.append({
        'TEST_ID': test_id,
        'QUESTION': question,
        'GENERATED_CYPHER': result['cypher'],
        'CYPHER_QUERY_RESULT': json.dumps(cypher_json_result, ensure_ascii=False) if cypher_json_result else '',
        'FORMATTED_CYPHER_QUERY_RESULT': formatted_cypher_result,
        'ANSWER': result['answer'],
        'STATUS': result['status'],
        'REFERENCES': result['references'],
        'TIME_S': result['time_s'],
    })
    
    # Delay between requests
    if idx < total - 1:
        time.sleep(DELAY_BETWEEN_REQUESTS)

results_df = pd.DataFrame(results)
print(f'\n=== Done: {len(results)} test cases evaluated ===')


Real-time evaluation logs will be written to: data/evaluation/e2e_eval_log_Q2C_004.txt

[1/2] HOTS_050: Seseorang yang nama baiknya tercemar akibat pemberitaan masa lalu yang tidak terbukti bersalah di pengadilan menuntut sebuah mesin pencari (search engine) untuk menghapus tautan berita tersebut dari hasil pencarian. Apakah tuntutan penghapusan informasi elektronik tersebut sah secara hukum?
    Status: ✅ PASS (31.8s)
    Generated Cypher:
    --------------------------------------------------------------------------------
    MATCH (n)
    WHERE (n:Pasal OR n:Ayat OR n:KonsepHukum OR n:PerbuatanHukum OR n:EntitasHukum)
      AND n.source_document_id IN ['POJK_11_2022', 'UU_11_2008', 'UU_19_2016']
      AND (
        toLower(n.content) CONTAINS 'hapus' OR toLower(n.content) CONTAINS 'penghapusan' OR
        toLower(n.content) CONTAINS 'informasi elektronik' OR toLower(n.content) CONTAINS 'sistem elektronik' OR
        toLower(n.content) CONTAINS 'pencemaran nama baik' OR toLower(n.con

## Step 5: Summary

In [ ]:
# === Overall Summary ===
total = len(results_df)
passed = len(results_df[results_df['STATUS'] == 'PASS'])
failed = len(results_df[results_df['STATUS'] == 'FAIL'])
errors = len(results_df[results_df['STATUS'] == 'ERROR'])
partial = len(results_df[results_df['STATUS'] == 'PARTIAL'])
avg_time = results_df['TIME_S'].mean()

print(f'╔══════════════════════════════════════════╗')
print(f'║  Q2C E2E Evaluation Report               ║')
print(f'╠══════════════════════════════════════════╣')
print(f'║  Experiment: {EXPERIMENT_ID:<27s} ║')
print(f'║  Documents:  {", ".join(DOCUMENT_IDS):<27s} ║')
print(f'║  Backend:    {BACKEND_URL:<27s} ║')
print(f'╠══════════════════════════════════════════╣')
print(f'║  Total:   {total:>3d}                              ║')
print(f'║  Pass:    {passed:>3d}  ({passed/total:.1%})                     ║')
print(f'║  Fail:    {failed:>3d}  ({failed/total:.1%})                     ║')
print(f'║  Error:   {errors:>3d}                              ║')
print(f'║  Partial: {partial:>3d}                              ║')
print(f'║  Avg Time: {avg_time:.1f}s                          ║')
print(f'╚══════════════════════════════════════════╝')

╔══════════════════════════════════════════╗
║  Q2C E2E Evaluation Report               ║
╠══════════════════════════════════════════╣
║  Experiment: Q2C_004                     ║
║  Documents:  POJK_11_2022, UU_11_2008, UU_19_2016 ║
║  Backend:    http://localhost:8000       ║
╠══════════════════════════════════════════╣
║  Total:     2                              ║
║  Pass:      2  (100.0%)                     ║
║  Fail:      0  (0.0%)                     ║
║  Error:     0                              ║
║  Partial:   0                              ║
║  Avg Time: 37.7s                          ║
╚══════════════════════════════════════════╝


In [ ]:
# === Failed/Error Test Cases ===
failed_df = results_df[results_df['STATUS'] != 'PASS']

if len(failed_df) == 0:
    print('🎉 All test cases passed!')
else:
    print(f'\n❌ Failed/Error test cases ({len(failed_df)}):\n')
    for _, row in failed_df.iterrows():
        print(f"  {row['TEST_ID']}: {row['QUESTION'][:60]}")
        print(f"    Status: {row['STATUS']}")
        if row['GENERATED_CYPHER']:
            print(f"    Cypher: {row['GENERATED_CYPHER'][:100]}...")
        print()

🎉 All test cases passed!


## Step 6: Save Results

In [ ]:
# === Save to CSV ===
output_dir = 'data/evaluation'
os.makedirs(output_dir, exist_ok=True)

output_csv = f'{output_dir}/e2e_eval_{EXPERIMENT_ID}.csv'
results_df.to_csv(output_csv, index=False, encoding='utf-8')
print(f'✅ Results saved to: {output_csv}')

✅ Results saved to: data/evaluation/e2e_eval_Q2C_004.csv


In [ ]:
# === (Optional) Write to Google Sheets ===
if WRITE_TO_GSHEETS:
    from modules.google_sheets_utils import GoogleUtil, GoogleSheetsWriter
    import gspread

    gu = GoogleUtil(
        private_key=os.getenv('GOOGLE_SHEETS_PRIVATE_KEY', ''),
        client_email=os.getenv('GOOGLE_SHEETS_CLIENT_EMAIL', '')
    )
    spreadsheet_id = os.getenv('GOOGLE_SPREADSHEET_ID', '')

    # Auto-create worksheet if it does not exist
    try:
        info = gu._get_google_info(gu.private_key, gu.client_email)
        client = gu._get_client(info, gu.GOOGLE_SHEETS_SCOPES)
        sh = client.open_by_key(spreadsheet_id)
        try:
            ws = sh.worksheet(EXPERIMENT_SHEET_NAME)
            print(f'ℹ️  Sheet "{EXPERIMENT_SHEET_NAME}" already exists.')
        except gspread.exceptions.WorksheetNotFound:
            ws = sh.add_worksheet(title=EXPERIMENT_SHEET_NAME, rows=1000, cols=20)
            # Write header row
            headers = list(results_df.columns)
            ws.update([headers], "A1")
            print(f'✅ Created new sheet: "{EXPERIMENT_SHEET_NAME}"')
    except Exception as e:
        print(f'⚠️  Could not auto-create sheet: {e}')

    writer = GoogleSheetsWriter(
        google_util=gu,
        sheet_id=spreadsheet_id,
        worksheet_name=EXPERIMENT_SHEET_NAME,
        batch_size=5,
    )

    result = writer.write_dataframe(results_df)
    print(f'✅ Written to Google Sheets: {EXPERIMENT_SHEET_NAME}')
    print(f'   Success: {result.successful_rows}, Failed: {result.failed_rows}')
else:
    print('ℹ️  WRITE_TO_GSHEETS = False, skipping Google Sheets upload.')
    print(f'   Set WRITE_TO_GSHEETS = True to write results to sheet "{EXPERIMENT_SHEET_NAME}"')


✅ Created new sheet: "EXP_E2E_HOTS_Q2C_TEST_DATA_V3_MISSING_POJK_11_2022_UU_11_2008_UU_19_2016"


  0%|          | 0/1 [00:00<?, ?it/s]2026-06-01 08:30:27,412 - WARNING - Transient error (APIError: [400]: Your input contains more than the maximum o). Waiting 2.95s before retry 1/5
2026-06-01 08:30:33,296 - WARNING - Transient error (APIError: [400]: Your input contains more than the maximum o). Waiting 4.58s before retry 2/5
2026-06-01 08:30:40,716 - WARNING - Transient error (APIError: [400]: Your input contains more than the maximum o). Waiting 8.04s before retry 3/5
2026-06-01 08:30:51,316 - WARNING - Transient error (APIError: [400]: Your input contains more than the maximum o). Waiting 16.11s before retry 4/5
2026-06-01 08:31:10,249 - ERROR - Failed after 5 attempts for row: {'TEST_ID': 'HOTS_050', 'QUESTION': 'Seseorang yang nama baiknya tercemar akibat pemberitaan masa lalu yang tidak terbukti bersalah di pengadilan menuntut sebuah mesin pencari (search engine) untuk menghapus tautan berita tersebut dari hasil pencarian. Apakah tuntutan penghapusan informasi elektronik terse

✅ Written to Google Sheets: EXP_E2E_HOTS_Q2C_TEST_DATA_V3_MISSING_POJK_11_2022_UU_11_2008_UU_19_2016
   Success: 0, Failed: 2


In [ ]:
# === Preview Results ===
print(f"\n{'='*80}")
print(f"RESULTS PREVIEW — First 5 rows")
print(f"{'='*80}")

for _, row in results_df.head(5).iterrows():
    print(f"\n--- {row['TEST_ID']} [{row['STATUS']}] ---")
    print(f"Q: {row['QUESTION']}")
    print(f"Cypher: {row['GENERATED_CYPHER'][:150]}..." if len(str(row['GENERATED_CYPHER'])) > 150 else f"Cypher: {row['GENERATED_CYPHER']}")
    print(f"Neo4j Result: {str(row['CYPHER_QUERY_RESULT'])[:200]}..." if len(str(row['CYPHER_QUERY_RESULT'])) > 200 else f"Neo4j Result: {row['CYPHER_QUERY_RESULT']}")
    print(f"Answer: {str(row['ANSWER'])[:200]}..." if len(str(row['ANSWER'])) > 200 else f"Answer: {row['ANSWER']}")


RESULTS PREVIEW — First 5 rows

--- HOTS_050 [PASS] ---
Q: Seseorang yang nama baiknya tercemar akibat pemberitaan masa lalu yang tidak terbukti bersalah di pengadilan menuntut sebuah mesin pencari (search engine) untuk menghapus tautan berita tersebut dari hasil pencarian. Apakah tuntutan penghapusan informasi elektronik tersebut sah secara hukum?
Cypher: MATCH (n)
WHERE (n:Pasal OR n:Ayat OR n:KonsepHukum OR n:PerbuatanHukum OR n:EntitasHukum)
  AND n.source_document_id IN ['POJK_11_2022', 'UU_11_2008'...
Neo4j Result: [{"pasal_ayat_label": "Agen Elektronik", "isi": "perangkat dari suatu Sistem Elektronik yang dibuat untuk melakukan suatu tindakan terhadap suatu Informasi Elektronik tertentu secara otomatis yang dis...
Answer: Berdasarkan data yang tersedia, tuntutan penghapusan informasi elektronik tersebut dapat sah secara hukum, dengan syarat adanya penetapan pengadilan.

Berikut adalah dasar hukumnya:

**Menurut UU No. ...

--- HOTS_088 [PASS] ---
Q: Sebuah bank umum mengalami k

In [ ]:
print('\n✅ Evaluation complete.')


✅ Evaluation complete.
